In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-6-0-clock

Analyze age-prediction clock models and plots.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


# 1. Aging clock + CR direct transfer

In [ ]:

# ============================================================
# mlp_tanh-only mouse aging clock + rat CR direct transfer

# ============================================================
import os
import sys
import glob
import numpy as np
import pandas as pd
import h5py
import joblib
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr, ttest_ind
from tqdm import tqdm
import scipy.sparse as sp
import scipy.io
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================
# 1. plotting
# ============================================================
sc.settings.set_figure_params(dpi=300, facecolor="white", format="pdf")
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "sans-serif"]
mpl.rcParams["font.size"] = 7
mpl.rcParams["axes.titlesize"] = 8
mpl.rcParams["axes.labelsize"] = 7
mpl.rcParams["xtick.labelsize"] = 7
mpl.rcParams["ytick.labelsize"] = 7
mpl.rcParams["legend.fontsize"] = 6
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

COLOR_DOTS = "#000000"
COLOR_LINE = "#9C27B0"
COLOR_IDENTITY = "#333333"
RANDOM_SEED = 42
COND_COLORS = {
    "Y": "#61A48F",
    "O": "#EA8D74",
    "CR": "#8798C4",
}

# ============================================================
# 2. paths and constants
# ============================================================
H5_BASE_DIR = input_path("1-TMS-remove/2-restart")
HEADER_FILE = input_path("header.txt")
GENE_LIST_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
RAT_DATA_DIR = input_path("2-8.3-shanda/1-data/2-valid/GSE137869_RAW")
ORTHOLOG_FILE = input_path("2-8.3-shanda/1-feature/mouse_rat_orthologs.csv")

OUTPUT_ROOT_DIR = output_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/4.4-CR-mlp-tanh-only-FigB-color")
MODEL_SAVE_DIR = os.path.join(OUTPUT_ROOT_DIR, "models", "mlp_tanh")
INTERNAL_CLOCK_PLOT_DIR = os.path.join(OUTPUT_ROOT_DIR, "internal_clock_plots", "mlp_tanh")
EXTERNAL_CR_PLOT_DIR = os.path.join(OUTPUT_ROOT_DIR, "external_cr_violin", "mlp_tanh")
SUMMARY_DIR = os.path.join(OUTPUT_ROOT_DIR, "summaries")

for d in [MODEL_SAVE_DIR, INTERNAL_CLOCK_PLOT_DIR, EXTERNAL_CR_PLOT_DIR, SUMMARY_DIR]:
    os.makedirs(d, exist_ok=True)

AGE_MAPPING = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}
TISSUE_MAPPING = {
    "Liver": "Liver",
    "Kidney": "Kidney",
    "Skin": "Skin",
    "Limb_Muscle": "Muscle",
    "Marrow": "BM",
}

MLP_TANH_PARAMS = {
    "hidden_layer_sizes": (64, 32),
    "activation": "tanh",
    "alpha": 0.01,
    "max_iter": 500,
    "early_stopping": True,
    "random_state": 42,
}

# ============================================================
# 3. utilities
# ============================================================
def normalize_gene_symbol(gene):
    if pd.isna(gene):
        return ""
    return str(gene).strip().upper()


def map_labels_to_months(labels, mapping):
    return np.vectorize(mapping.get)(labels)


def get_gene_names():
    with open(HEADER_FILE, "r") as f:
        return [line.strip().upper() for line in f if line.strip()]


def read_table_auto(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if ext in [".tsv", ".txt"]:
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)


def find_column(columns, keywords):
    cols = list(columns)
    lower = {c: str(c).lower() for c in cols}
    for c in cols:
        name = lower[c]
        if all(k in name for k in keywords):
            return c
    return None


def load_mouse_rat_ortholog_map(ortholog_file):
    if not os.path.exists(ortholog_file):
        raise FileNotFoundError(f"未找到小鼠-大鼠同源基因表: {ortholog_file}")

    df = read_table_auto(ortholog_file)
    mouse_col = find_column(df.columns, ["mouse", "gene"])
    rat_col = find_column(df.columns, ["rat", "gene"])

    if mouse_col is None:
        mouse_col = find_column(df.columns, ["mouse", "symbol"])
    if rat_col is None:
        rat_col = find_column(df.columns, ["rat", "symbol"])

    if mouse_col is None or rat_col is None:
        raise ValueError(
            "ortholog 表必须包含 mouse gene symbol 和 rat gene symbol 两列。"
        )

    out = {}
    for _, row in df.iterrows():
        mouse_gene = normalize_gene_symbol(row[mouse_col])
        rat_gene = normalize_gene_symbol(row[rat_col])
        if mouse_gene and rat_gene:
            out.setdefault(mouse_gene, [])
            if rat_gene not in out[mouse_gene]:
                out[mouse_gene].append(rat_gene)
    return out


def choose_rat_feature(mouse_gene, ortholog_map, rat_var_lookup):
    """Prefer the curated ortholog; fall back to the same symbol if absent from the rat matrix."""
    mouse_key = normalize_gene_symbol(mouse_gene)
    candidates = ortholog_map.get(mouse_key, [])
    for rat_gene in candidates:
        rat_key = normalize_gene_symbol(rat_gene)
        if rat_key in rat_var_lookup:
            return rat_var_lookup[rat_key], rat_gene, candidates, "ortholog_table"


    fallback_candidates = [mouse_key, mouse_key.capitalize()]
    for rat_gene in fallback_candidates:
        rat_key = normalize_gene_symbol(rat_gene)
        if rat_key in rat_var_lookup:
            return rat_var_lookup[rat_key], rat_gene, candidates, "symbol_fallback"

    if candidates:
        return None, candidates[0], candidates, "ortholog_table_missing_in_rat_matrix"
    return None, None, [], "no_ortholog"


def safe_ttest_greater(group1, group2):
    group1 = pd.Series(group1).dropna()
    group2 = pd.Series(group2).dropna()
    if len(group1) < 2 or len(group2) < 2:
        return np.nan
    try:
        _, p_val = ttest_ind(group1, group2, alternative="greater", equal_var=False)
        return float(p_val)
    except Exception:
        return np.nan


def safe_ttest_less(group1, group2):
    group1 = pd.Series(group1).dropna()
    group2 = pd.Series(group2).dropna()
    if len(group1) < 2 or len(group2) < 2:
        return np.nan
    try:
        _, p_val = ttest_ind(group1, group2, alternative="less", equal_var=False)
        return float(p_val)
    except Exception:
        return np.nan


def format_p_value(p_val):
    if pd.isna(p_val):
        return "n.s."
    if p_val >= 0.05:
        return "n.s."
    if p_val < 2.2e-16:
        return r"$P$ < 2.2e-16"
    return rf"$P$ = {p_val:.1e}"


def add_stat_bracket(ax, x1, x2, y, h, p_val):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=0.6, c="black", clip_on=False)
    ax.text((x1 + x2) * 0.5, y + h * 1.15, format_p_value(p_val), ha="center", va="bottom", fontsize=7)


def set_closed_box(ax, linewidth=0.6):
    for side in ["top", "right", "bottom", "left"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(linewidth)
    ax.tick_params(width=linewidth, length=3)


def evaluate_age_prediction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if len(y_true) > 1 and len(np.unique(y_pred)) > 1:
        r, p = pearsonr(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
    else:
        r, p, mae = np.nan, np.nan, np.nan
    return r, p, mae


def build_mlp_tanh_model():
    return make_pipeline(
        StandardScaler(),
        MLPRegressor(**MLP_TANH_PARAMS)
    )

# ============================================================
# 4. internal clock plot
# ============================================================
def plot_internal_clock_scatter(tissue, y_test, y_pred, output_prefix):
    """Draw the internal age-clock scatter plot with single-column size and adaptive axis margins."""
    y_test = np.asarray(y_test, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid_mask = np.isfinite(y_test) & np.isfinite(y_pred)
    y_test_plot = y_test[valid_mask]
    y_pred_plot = y_pred[valid_mask]

    if len(y_test_plot) == 0:
        return np.nan, np.nan, np.nan


    all_vals = np.concatenate([y_test_plot, y_pred_plot])
    axis_min = 0.0
    axis_max = max(35.0, float(np.nanmax(all_vals)) + 3.0)
    axis_max = float(np.ceil(axis_max / 5.0) * 5.0)

    fig_w = 42 / 25.4
    fig_h = 42 / 25.4
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))


    ax.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        ls="--",
        c=COLOR_IDENTITY,
        lw=0.8,
        alpha=0.9,
        zorder=0,
    )


    sns.regplot(
        x=y_test_plot,
        y=y_pred_plot,
        ax=ax,
        scatter=False,
        line_kws={"color": COLOR_LINE, "linewidth": 1.4},
        ci=95,
        truncate=False,
    )
    for collection in ax.collections:
        if isinstance(collection, mpl.collections.PolyCollection):
            collection.set_alpha(0.16)
            collection.set_facecolor("#A6A6A6")
            collection.set_edgecolor("none")


    rng = np.random.default_rng(RANDOM_SEED)
    x_jittered = y_test_plot + rng.normal(0, 0.35, size=len(y_test_plot))
    ax.scatter(
        x_jittered,
        y_pred_plot,
        s=4.2,
        c=COLOR_DOTS,
        alpha=0.38,
        edgecolors="none",
        linewidths=0,
        rasterized=True,
        zorder=2,
        clip_on=False,
    )

    r, p, mae = evaluate_age_prediction(y_test_plot, y_pred_plot)
    if pd.isna(p):
        p_str = "n.a."
    else:
        p_str = "< 2.2e-16" if p < 2.2e-16 else f"= {p:.1e}"
    stats_text = (
        rf"R = {r:.2f}" + "\n" +
        rf"$\mathit{{P}}$ {p_str}" + "\n" +
        rf"MAE = {mae:.2f}"
    )
    ax.text(
        0.055,
        0.945,
        stats_text,
        transform=ax.transAxes,
        fontsize=7,
        va="top",
        ha="left",
        color="black",
        linespacing=1.25,
    )

    display_title = tissue.replace("_", " ")
    ax.set_title(display_title, pad=5, fontweight="normal", fontsize=8)
    ax.set_xlabel("Chronological age (months)", fontsize=7)
    ax.set_ylabel("Predicted age (months)", fontsize=7)
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)

    ticks = [t for t in [0, 10, 20, 30, 40] if axis_min <= t <= axis_max]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis="both", which="major", direction="out", length=2.5, width=0.6, pad=1.5, labelsize=6)

    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
        spine.set_color("black")

    fig.tight_layout(pad=0.35)
    plt.savefig(output_prefix + ".pdf", bbox_inches="tight", pad_inches=0.03)
    plt.savefig(output_prefix + ".png", bbox_inches="tight", pad_inches=0.03, dpi=600)
    plt.close(fig)
    return r, p, mae

# ============================================================
# 5. train mouse mlp_tanh clocks
# ============================================================
def train_mouse_mlp_tanh_clocks():
    print("\n" + "=" * 80)
    print("Step 1: 使用 Final SAGE genes 构建 mouse mlp_tanh aging clocks")
    print("=" * 80)

    real_gene_names = get_gene_names()
    txt_files = [f for f in os.listdir(GENE_LIST_DIR) if f.endswith(".txt") and "Knee_Genes" in f]
    train_rows = []
    missing_rows = []

    for txt_file in tqdm(txt_files, desc="Training mlp_tanh clocks"):
        tissue = txt_file.split("_Knee_Genes")[0]
        gene_file = os.path.join(GENE_LIST_DIR, txt_file)
        with open(gene_file, "r") as f:
            selected_genes = [normalize_gene_symbol(line) for line in f if line.strip()]

        train_h5 = os.path.join(H5_BASE_DIR, tissue, "train.h5")
        test_h5 = os.path.join(H5_BASE_DIR, tissue, "test.h5")
        if not (os.path.exists(train_h5) and os.path.exists(test_h5)):
            continue

        with h5py.File(train_h5, "r") as f:
            X_train_all = f["data"][:].astype(np.float32)
            y_train = map_labels_to_months(f["label"][:, 0].astype(np.int8), AGE_MAPPING)
        with h5py.File(test_h5, "r") as f:
            X_test_all = f["data"][:].astype(np.float32)
            y_test = map_labels_to_months(f["label"][:, 0].astype(np.int8), AGE_MAPPING)

        df_train = pd.DataFrame(X_train_all, columns=real_gene_names)
        df_test = pd.DataFrame(X_test_all, columns=real_gene_names)

        valid_genes = []
        used = set()
        for gene in selected_genes:
            if gene in used:
                continue
            if gene in df_train.columns:
                valid_genes.append(gene)
                used.add(gene)
            else:
                missing_rows.append({"tissue": tissue, "gene": gene, "reason": "not_in_mouse_h5_header"})

        if len(valid_genes) < 2:
            continue

        model = build_mlp_tanh_model()
        model.fit(df_train[valid_genes].values, y_train)
        y_pred = model.predict(df_test[valid_genes].values)

        safe_tissue = tissue.replace(" ", "_").replace("/", "-")
        model_path = os.path.join(MODEL_SAVE_DIR, f"{safe_tissue}__mlp_tanh_Clock.pkl")
        internal_prefix = os.path.join(INTERNAL_CLOCK_PLOT_DIR, f"{safe_tissue}__mlp_tanh_internal_scatter")
        r, p, mae = plot_internal_clock_scatter(tissue, y_test, y_pred, internal_prefix)

        feature_means = df_train[valid_genes].mean(axis=0).values.astype(np.float32)
        joblib.dump(
            {
                "model": model,
                "features": valid_genes,
                "feature_means": feature_means,
                "model_name": "mlp_tanh",
                "model_class": "Pipeline(StandardScaler+MLPRegressor)",
                "mlp_tanh_params": MLP_TANH_PARAMS,
                "feature_policy": "Final SAGE genes, no additional feature selection",
                "source_gene_file": gene_file,
                "base_sage_gene_count": len(selected_genes),
                "valid_sage_gene_count": len(valid_genes),
                "tissue": tissue,
            },
            model_path,
        )

        train_rows.append(
            {
                "tissue": tissue,
                "model_name": "mlp_tanh",
                "model_path": model_path,
                "source_gene_file": gene_file,
                "base_sage_gene_count": len(selected_genes),
                "valid_sage_gene_count": len(valid_genes),
                "n_train": len(y_train),
                "n_test": len(y_test),
                "test_PCC": r,
                "test_P_value": p,
                "test_MAE": mae,
                "internal_scatter_pdf": internal_prefix + ".pdf",
                "internal_scatter_png": internal_prefix + ".png",
            }
        )

    train_df = pd.DataFrame(train_rows)
    train_path = os.path.join(SUMMARY_DIR, "Mouse_mlp_tanh_Clock_Training_Summary.csv")
    train_df.to_csv(train_path, index=False)
    pd.DataFrame(missing_rows).to_csv(os.path.join(SUMMARY_DIR, "Mouse_mlp_tanh_Missing_Mouse_Matrix_Genes.csv"), index=False)
    print(f"Mouse mlp_tanh clocks saved: {train_path}")
    return train_df

# ============================================================
# 6. load and preprocess rat CR data
# ============================================================
def load_custom_10x(prefix, dir_path):
    mat_path = os.path.join(dir_path, prefix + "_matrix.mtx.gz")
    bc_path = os.path.join(dir_path, prefix + "_barcodes.tsv.gz")
    gene_path = os.path.join(dir_path, prefix + "_genes.tsv.gz")
    if not (os.path.exists(mat_path) and os.path.exists(bc_path) and os.path.exists(gene_path)):
        return None
    adata = ad.AnnData(X=scipy.io.mmread(mat_path).T.tocsr())
    adata.obs_names = pd.read_csv(bc_path, header=None, sep="\t")[0].values
    genes_df = pd.read_csv(gene_path, header=None, sep="\t")
    adata.var_names = genes_df[1].values if genes_df.shape[1] > 1 else genes_df[0].values
    adata.var_names_make_unique()
    return adata


def parse_condition_from_prefix(prefix):
    try:
        for token in prefix.split("_")[1].split("-"):
            if token in ["Y", "O", "CR"]:
                return token
    except Exception:
        return None
    return None


def load_rat_tissue_data(rat_tissue_name):
    mat_files = glob.glob(os.path.join(RAT_DATA_DIR, f"*_{rat_tissue_name}-*_matrix.mtx.gz"))
    adatas = []
    for file_path in mat_files:
        prefix = os.path.basename(file_path).replace("_matrix.mtx.gz", "")
        condition = parse_condition_from_prefix(prefix)
        if condition not in ["Y", "O", "CR"]:
            continue
        adata_tmp = load_custom_10x(prefix, RAT_DATA_DIR)
        if adata_tmp is None:
            continue
        adata_tmp.obs["Condition"] = condition
        adata_tmp.obs["Sample"] = prefix
        adata_tmp.obs_names = [f"{prefix}_{bc}" for bc in adata_tmp.obs_names]
        adatas.append(adata_tmp)
    if not adatas:
        return None
    adata = ad.concat(adatas, join="outer")
    adata.obs_names_make_unique()
    adata.var_names_make_unique()
    return adata


def preprocess_rat_adata(adata):
    adata.var["mt"] = adata.var_names.str.startswith("mt-") | adata.var_names.str.startswith("Mt-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True)
    sc.pp.filter_cells(adata, min_genes=200)
    adata = adata[adata.obs["n_genes_by_counts"] > 500].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.obs["Condition"] = pd.Categorical(adata.obs["Condition"], categories=["Y", "O", "CR"], ordered=True)
    return adata

# ============================================================
# 7. external feature matrix and plot
# ============================================================
def build_external_feature_matrix(adata, model_package, ortholog_map):
    features = model_package["features"]
    feature_means = np.asarray(model_package["feature_means"], dtype=np.float32)
    X_target = np.tile(feature_means, (adata.n_obs, 1)).astype(np.float32)
    rat_var_lookup = {normalize_gene_symbol(g): g for g in adata.var_names}

    matched_rows = []
    missing_rows = []
    for i, mouse_gene in enumerate(features):
        rat_gene_used, rat_gene, candidates, source = choose_rat_feature(mouse_gene, ortholog_map, rat_var_lookup)
        if rat_gene_used is None:
            missing_rows.append(
                {
                    "mouse_clock_gene": mouse_gene,
                    "feature_index": i,
                    "rat_ortholog_gene": rat_gene,
                    "rat_ortholog_candidates": ";".join(candidates),
                    "match_source": source,
                    "reason": "not_found_in_external_rat_matrix",
                    "imputation": "mouse_training_feature_mean",
                }
            )
            continue
        expr = adata[:, rat_gene_used].X
        X_target[:, i] = (expr.toarray().reshape(-1) if sp.issparse(expr) else np.asarray(expr).reshape(-1)).astype(np.float32)
        matched_rows.append(
            {
                "mouse_clock_gene": mouse_gene,
                "feature_index": i,
                "rat_ortholog_gene": rat_gene,
                "rat_ortholog_candidates": ";".join(candidates),
                "rat_gene_used": rat_gene_used,
                "match_source": source,
                "imputation": "not_imputed",
            }
        )
    return X_target, matched_rows, missing_rows


def plot_external_predicted_age(adata, rat_tissue_name, output_prefix):
    plot_df = adata.obs[["Condition", "Sample", "Predicted_Age"]].copy().dropna()
    plot_df["Condition"] = pd.Categorical(plot_df["Condition"], categories=["Y", "O", "CR"], ordered=True)

    sample_df = (
        plot_df.groupby(["Sample", "Condition"], observed=True)["Predicted_Age"]
        .median()
        .reset_index()
    )
    sample_df["Condition"] = pd.Categorical(sample_df["Condition"], categories=["Y", "O", "CR"], ordered=True)

    y_cell = plot_df.loc[plot_df["Condition"] == "Y", "Predicted_Age"]
    o_cell = plot_df.loc[plot_df["Condition"] == "O", "Predicted_Age"]
    cr_cell = plot_df.loc[plot_df["Condition"] == "CR", "Predicted_Age"]
    y_sample = sample_df.loc[sample_df["Condition"] == "Y", "Predicted_Age"]
    o_sample = sample_df.loc[sample_df["Condition"] == "O", "Predicted_Age"]
    cr_sample = sample_df.loc[sample_df["Condition"] == "CR", "Predicted_Age"]


    p_aging_cell = safe_ttest_greater(o_cell, y_cell)
    p_treat_cell = safe_ttest_less(cr_cell, o_cell)
    p_aging_sample = safe_ttest_greater(o_sample, y_sample)
    p_treat_sample = safe_ttest_less(cr_sample, o_sample)

    o_minus_y_cell = o_cell.mean() - y_cell.mean() if len(o_cell) and len(y_cell) else np.nan
    cr_minus_o_cell = cr_cell.mean() - o_cell.mean() if len(cr_cell) and len(o_cell) else np.nan
    o_minus_y_sample = o_sample.mean() - y_sample.mean() if len(o_sample) and len(y_sample) else np.nan
    cr_minus_o_sample = cr_sample.mean() - o_sample.mean() if len(cr_sample) and len(o_sample) else np.nan

    fig, ax = plt.subplots(figsize=(2.4, 2.4))
    sns.violinplot(
        x="Condition", y="Predicted_Age", data=plot_df, order=["Y", "O", "CR"],
        palette=[COND_COLORS["Y"], COND_COLORS["O"], COND_COLORS["CR"]],
        inner=None, linewidth=0.5, width=0.78, cut=0, saturation=1.0, ax=ax,
    )
    sns.boxplot(
        x="Condition", y="Predicted_Age", data=plot_df, order=["Y", "O", "CR"],
        color="white", width=0.12, fliersize=0, zorder=2, ax=ax,
        boxprops={"edgecolor": "black", "linewidth": 0.6},
        medianprops={"color": "black", "linewidth": 0.8},
        whiskerprops={"color": "black", "linewidth": 0.6},
        capprops={"color": "black", "linewidth": 0.6},
    )
    sns.stripplot(
        x="Condition", y="Predicted_Age", data=sample_df, order=["Y", "O", "CR"],
        color="black", size=2.4, jitter=0.08, alpha=0.85, zorder=4, ax=ax,
    )

    y_ref = y_cell.median() if len(y_cell) else plot_df["Predicted_Age"].median()
    ax.axhline(y_ref, color="grey", linestyle="--", linewidth=0.6, alpha=0.75, zorder=0)
    y_low, y_high = plot_df["Predicted_Age"].quantile([0.005, 0.995])
    y_span = y_high - y_low
    if y_span <= 0:
        y_span = 1.0
    h = y_span * 0.055
    add_stat_bracket(ax, 0, 1, y_high + h * 2.0, h * 1.2, p_aging_cell)
    add_stat_bracket(ax, 1, 2, y_high + h * 4.6, h * 1.2, p_treat_cell)
    ax.set_ylim(y_low - y_span * 0.12, y_high + y_span * 0.34)
    ax.set_title(f"Rat {rat_tissue_name} | mlp_tanh\nCR-Old = {cr_minus_o_cell:+.2f} mo", fontsize=8)
    ax.set_ylabel("Predicted age (months)")
    ax.set_xlabel("")
    ax.set_xticklabels(["Y", "O", "CR"])
    set_closed_box(ax)
    plt.tight_layout()

    pdf_path = output_prefix + ".pdf"
    png_path = output_prefix + ".png"
    plt.savefig(pdf_path)
    plt.savefig(png_path, dpi=600)
    plt.close(fig)

    plot_df.to_csv(output_prefix + "_cell_level_predicted_age.csv", index=True)
    sample_df.to_csv(output_prefix + "_sample_level_predicted_age.csv", index=False)
    (
        plot_df.groupby("Condition", observed=True)["Predicted_Age"]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
        .to_csv(output_prefix + "_group_level_predicted_age.csv", index=False)
    )

    return {
        "rat_tissue": rat_tissue_name,
        "n_cells_Y": int((plot_df["Condition"] == "Y").sum()),
        "n_cells_O": int((plot_df["Condition"] == "O").sum()),
        "n_cells_CR": int((plot_df["Condition"] == "CR").sum()),
        "n_samples_Y": int(len(y_sample)),
        "n_samples_O": int(len(o_sample)),
        "n_samples_CR": int(len(cr_sample)),
        "Y_cell_median": float(y_cell.median()) if len(y_cell) else np.nan,
        "O_cell_median": float(o_cell.median()) if len(o_cell) else np.nan,
        "CR_cell_median": float(cr_cell.median()) if len(cr_cell) else np.nan,
        "Y_sample_median": float(y_sample.median()) if len(y_sample) else np.nan,
        "O_sample_median": float(o_sample.median()) if len(o_sample) else np.nan,
        "CR_sample_median": float(cr_sample.median()) if len(cr_sample) else np.nan,
        "O_minus_Y_cell_mean": float(o_minus_y_cell) if pd.notna(o_minus_y_cell) else np.nan,
        "CR_minus_O_cell_mean": float(cr_minus_o_cell) if pd.notna(cr_minus_o_cell) else np.nan,
        "O_minus_Y_sample_mean": float(o_minus_y_sample) if pd.notna(o_minus_y_sample) else np.nan,
        "CR_minus_O_sample_mean": float(cr_minus_o_sample) if pd.notna(cr_minus_o_sample) else np.nan,
        "p_aging_cell_level": float(p_aging_cell) if pd.notna(p_aging_cell) else np.nan,
        "p_treat_cell_level": float(p_treat_cell) if pd.notna(p_treat_cell) else np.nan,
        "p_aging_sample_level": float(p_aging_sample) if pd.notna(p_aging_sample) else np.nan,
        "p_treat_sample_level": float(p_treat_sample) if pd.notna(p_treat_sample) else np.nan,
        "significance_test": "Welch t-test, one-sided, cell-level, same as rapamycin notebook",
        "external_violin_pdf": pdf_path,
        "external_violin_png": png_path,
        "cell_level_predicted_age_csv": output_prefix + "_cell_level_predicted_age.csv",
        "sample_level_predicted_age_csv": output_prefix + "_sample_level_predicted_age.csv",
        "group_level_predicted_age_csv": output_prefix + "_group_level_predicted_age.csv",
    }

# ============================================================
# 8. external rat CR validation
# ============================================================
def validate_mlp_tanh_on_rat_CR():
    print("\n" + "=" * 80)
    print("Step 2: 将 mouse mlp_tanh aging clocks 迁移到 rat CR 外部数据")
    print("=" * 80)

    ortholog_map = load_mouse_rat_ortholog_map(ORTHOLOG_FILE)
    rows = []

    for mouse_tissue, rat_tissue in TISSUE_MAPPING.items():
        safe_mouse_tissue = mouse_tissue.replace(" ", "_").replace("/", "-")
        model_path = os.path.join(MODEL_SAVE_DIR, f"{safe_mouse_tissue}__mlp_tanh_Clock.pkl")
        if not os.path.exists(model_path):
            print(f"缺少模型: {model_path}")
            continue

        model_package = joblib.load(model_path)
        model = model_package["model"]
        features = model_package["features"]

        adata = load_rat_tissue_data(rat_tissue)
        if adata is None:
            print(f"Rat {rat_tissue}: 未找到外部数据，跳过。")
            continue
        print(f"\n{mouse_tissue} -> Rat {rat_tissue}")
        print("Raw condition counts:")
        print(adata.obs["Condition"].value_counts())
        adata = preprocess_rat_adata(adata)
        print("After QC condition counts:")
        print(adata.obs["Condition"].value_counts())

        X_target, matched_rows, missing_rows = build_external_feature_matrix(adata, model_package, ortholog_map)
        adata.obs["Predicted_Age"] = model.predict(X_target)

        output_prefix = os.path.join(EXTERNAL_CR_PLOT_DIR, f"{mouse_tissue}__mlp_tanh__{rat_tissue}_boxplot")
        pd.DataFrame(matched_rows).to_csv(output_prefix + "_matched_genes.csv", index=False)
        pd.DataFrame(missing_rows).to_csv(output_prefix + "_missing_features_imputed_by_mouse_training_mean.csv", index=False)

        stats = plot_external_predicted_age(adata, rat_tissue, output_prefix)
        match_rate = len(matched_rows) / len(features) if features else np.nan
        stats.update(
            {
                "mouse_model": mouse_tissue,
                "model_name": "mlp_tanh",
                "model_path": model_path,
                "clock_gene_total": len(features),
                "clock_gene_matched": len(matched_rows),
                "clock_gene_match_rate": match_rate,
                "missing_external_feature_count": len(missing_rows),
                "missing_feature_imputation": "mouse_training_feature_mean",
                "base_sage_gene_count": model_package.get("base_sage_gene_count"),
                "valid_sage_gene_count": model_package.get("valid_sage_gene_count"),
                "feature_policy": model_package.get("feature_policy"),
                "external_transfer_policy": "mouse clock fixed; rat features aligned by ortholog/symbol fallback; missing features imputed by mouse training mean",
            }
        )
        rows.append(stats)
        pd.DataFrame([stats]).to_csv(output_prefix + "_stats.csv", index=False)

    external_df = pd.DataFrame(rows)
    if not external_df.empty:
        external_df["aging_direction_ok"] = external_df["O_minus_Y_cell_mean"] > 0
        external_df["treat_direction_ok"] = external_df["CR_minus_O_cell_mean"] < 0
        external_df["aging_pass"] = external_df["aging_direction_ok"] & (external_df["p_aging_cell_level"] < 0.05)
        external_df["treat_pass"] = external_df["treat_direction_ok"] & (external_df["p_treat_cell_level"] < 0.05)
        external_df["both_pass"] = external_df["aging_pass"] & external_df["treat_pass"]

    external_path = os.path.join(SUMMARY_DIR, "Rat_CR_mlp_tanh_External_PredictedAge_Summary.csv")
    external_df.to_csv(external_path, index=False)
    print(f"Rat CR mlp_tanh external summary saved: {external_path}")
    return external_df

# ============================================================
# 9. main
# ============================================================
def main():
    train_df = train_mouse_mlp_tanh_clocks()
    external_df = validate_mlp_tanh_on_rat_CR()
    return train_df, external_df


if __name__ == "__main__":
    train_df, external_df = main()
    display(train_df)
    display(external_df)


Age-clock analysis and plots.

In [ ]:
# ============================================================
# Mouse tissue-specific aging clock construction + internal plot
# mlp_tanh only

# ============================================================

import os
import numpy as np
import pandas as pd
import h5py
import joblib
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from tqdm import tqdm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================
# 1. paths
# ============================================================

H5_BASE_DIR = input_path("1-TMS-remove/2-restart")
HEADER_FILE = input_path("header.txt")
GENE_LIST_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")

OUTPUT_ROOT_DIR = output_path("2-8.3-shanda/1-feature/1-figure/internal_mouse_mlp_tanh_clock")
MODEL_SAVE_DIR = os.path.join(OUTPUT_ROOT_DIR, "models")
PLOT_DIR = os.path.join(OUTPUT_ROOT_DIR, "internal_clock_plots")
SUMMARY_DIR = os.path.join(OUTPUT_ROOT_DIR, "summaries")

for d in [MODEL_SAVE_DIR, PLOT_DIR, SUMMARY_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# 2. basic settings
# ============================================================

AGE_MAPPING = {
    0: 1,
    1: 3,
    2: 18,
    3: 21,
    4: 24,
    5: 30
}

RANDOM_SEED = 42

MLP_TANH_PARAMS = {
    "hidden_layer_sizes": (64, 32),
    "activation": "tanh",
    "alpha": 0.01,
    "max_iter": 500,
    "early_stopping": True,
    "random_state": RANDOM_SEED
}

# ============================================================
# 3. plotting style
# ============================================================

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "sans-serif"]
mpl.rcParams["font.size"] = 8
mpl.rcParams["axes.titlesize"] = 9
mpl.rcParams["axes.labelsize"] = 8
mpl.rcParams["xtick.labelsize"] = 7
mpl.rcParams["ytick.labelsize"] = 7
mpl.rcParams["legend.fontsize"] = 7
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

COLOR_DOTS = "#000000"
COLOR_LINE = "#9C27B0"
COLOR_IDENTITY = "#333333"

# ============================================================
# 4. utility functions
# ============================================================

def normalize_gene_symbol(gene):
    if pd.isna(gene):
        return ""
    return str(gene).strip().upper()


def map_labels_to_months(labels, mapping):
    return np.vectorize(mapping.get)(labels)


def get_gene_names():
    with open(HEADER_FILE, "r") as f:
        return [line.strip().upper() for line in f if line.strip()]


def evaluate_age_prediction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if len(y_true) > 1 and len(np.unique(y_pred)) > 1:
        r, p = pearsonr(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
    else:
        r, p, mae = np.nan, np.nan, np.nan

    return r, p, mae


def build_mlp_tanh_model():
    return make_pipeline(
        StandardScaler(),
        MLPRegressor(**MLP_TANH_PARAMS)
    )


def get_axis_limit(y_true, y_pred):
    """
    Set axis limits from actual and predicted ages.
    Leave room above the largest prediction instead of fixing ylim to 36.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    max_val = np.nanmax([np.nanmax(y_true), np.nanmax(y_pred)])
    min_val = np.nanmin([np.nanmin(y_true), np.nanmin(y_pred)])


    axis_min = 0


    axis_max = max_val * 1.15
    axis_max = np.ceil(axis_max / 5) * 5


    axis_max = max(axis_max, 35)

    return axis_min, axis_max


# ============================================================
# 5. internal clock plot
# ============================================================

def plot_internal_clock_scatter(tissue, y_test, y_pred, output_prefix):
    y_test = np.asarray(y_test)
    y_pred = np.asarray(y_pred)

    r, p, mae = evaluate_age_prediction(y_test, y_pred)

    axis_min, axis_max = get_axis_limit(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(2.8, 2.8))

    # identity line
    ax.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        ls="--",
        c=COLOR_IDENTITY,
        lw=1.0,
        alpha=0.9,
        zorder=0
    )

    # regression line
    sns.regplot(
        x=y_test,
        y=y_pred,
        ax=ax,
        scatter=False,
        ci=95,
        line_kws={
            "color": COLOR_LINE,
            "linewidth": 1.8,
            "zorder": 3
        }
    )

    # CI color
    for collection in ax.collections:
        if isinstance(collection, mpl.collections.PolyCollection):
            collection.set_alpha(0.18)
            collection.set_facecolor("#B0B0B0")

    # scatter with x jitter
    np.random.seed(RANDOM_SEED)
    x_jittered = y_test + np.random.normal(0, 0.35, size=len(y_test))

    ax.scatter(
        x_jittered,
        y_pred,
        s=5,
        c=COLOR_DOTS,
        alpha=0.45,
        edgecolors="none",
        zorder=2
    )

    # stats text
    if pd.isna(p):
        p_str = "n.a."
    else:
        p_str = "< 2.2e-16" if p < 2.2e-16 else f"= {p:.1e}"

    stats_text = (
        rf"R = {r:.2f}" + "\n" +
        rf"$\mathit{{P}}$ {p_str}" + "\n" +
        rf"MAE = {mae:.2f}"
    )

    ax.text(
        0.05,
        0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=8,
        verticalalignment="top",
        horizontalalignment="left",
        color="black",
        linespacing=1.4
    )

    display_title = tissue.replace("_", " ")

    ax.set_title(
        f"{display_title}\nmlp_tanh",
        pad=8,
        fontsize=9,
        fontweight="normal"
    )

    ax.set_xlabel("Chronological age (months)")
    ax.set_ylabel("Predicted age (months)")


    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)


    ax.set_aspect("equal", adjustable="box")


    ticks = np.arange(0, axis_max + 1, 10)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)

    # closed box
    for side in ["top", "right", "bottom", "left"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.8)

    ax.tick_params(width=0.8, length=3)

    plt.tight_layout()

    plt.savefig(output_prefix + ".pdf", bbox_inches="tight", pad_inches=0.05)
    plt.savefig(output_prefix + ".png", bbox_inches="tight", pad_inches=0.05, dpi=600)

    plt.close(fig)

    return r, p, mae, axis_max


# ============================================================
# 6. train mouse clocks and plot internal validation
# ============================================================

def train_mouse_mlp_tanh_clocks():
    print("\n" + "=" * 80)
    print("Constructing mouse tissue-specific mlp_tanh aging clocks")
    print("=" * 80)

    real_gene_names = get_gene_names()

    txt_files = [
        f for f in os.listdir(GENE_LIST_DIR)
        if f.endswith(".txt") and "Knee_Genes" in f
    ]

    summary_rows = []
    missing_rows = []

    for txt_file in tqdm(txt_files, desc="Training mouse clocks"):

        tissue = txt_file.split("_Knee_Genes")[0]

        gene_file = os.path.join(GENE_LIST_DIR, txt_file)

        with open(gene_file, "r") as f:
            selected_genes = [
                normalize_gene_symbol(line)
                for line in f
                if line.strip()
            ]

        train_h5 = os.path.join(H5_BASE_DIR, tissue, "train.h5")
        test_h5 = os.path.join(H5_BASE_DIR, tissue, "test.h5")

        if not os.path.exists(train_h5):
            print(f"Skip {tissue}: train.h5 not found")
            continue

        if not os.path.exists(test_h5):
            print(f"Skip {tissue}: test.h5 not found")
            continue

        # ----------------------------
        # load train data
        # ----------------------------
        with h5py.File(train_h5, "r") as f:
            X_train_all = f["data"][:].astype(np.float32)
            y_train = map_labels_to_months(
                f["label"][:, 0].astype(np.int8),
                AGE_MAPPING
            )

        # ----------------------------
        # load test data
        # ----------------------------
        with h5py.File(test_h5, "r") as f:
            X_test_all = f["data"][:].astype(np.float32)
            y_test = map_labels_to_months(
                f["label"][:, 0].astype(np.int8),
                AGE_MAPPING
            )

        df_train = pd.DataFrame(X_train_all, columns=real_gene_names)
        df_test = pd.DataFrame(X_test_all, columns=real_gene_names)

        # ----------------------------
        # align SAGE genes
        # ----------------------------
        valid_genes = []
        used = set()

        for gene in selected_genes:
            if gene in used:
                continue

            if gene in df_train.columns:
                valid_genes.append(gene)
                used.add(gene)
            else:
                missing_rows.append({
                    "tissue": tissue,
                    "gene": gene,
                    "reason": "not_in_mouse_h5_header"
                })

        if len(valid_genes) < 2:
            print(f"Skip {tissue}: valid genes < 2")
            continue

        X_train = df_train[valid_genes].values
        X_test = df_test[valid_genes].values

        # ----------------------------
        # train model
        # ----------------------------
        model = build_mlp_tanh_model()
        model.fit(X_train, y_train)

        # ----------------------------
        # internal prediction
        # ----------------------------
        y_pred = model.predict(X_test)

        # ----------------------------
        # save model
        # ----------------------------
        safe_tissue = tissue.replace(" ", "_").replace("/", "-")

        model_path = os.path.join(
            MODEL_SAVE_DIR,
            f"{safe_tissue}__mlp_tanh_Clock.pkl"
        )

        feature_means = df_train[valid_genes].mean(axis=0).values.astype(np.float32)

        joblib.dump(
            {
                "model": model,
                "features": valid_genes,
                "feature_means": feature_means,
                "model_name": "mlp_tanh",
                "model_class": "Pipeline(StandardScaler + MLPRegressor)",
                "mlp_tanh_params": MLP_TANH_PARAMS,
                "source_gene_file": gene_file,
                "base_sage_gene_count": len(selected_genes),
                "valid_sage_gene_count": len(valid_genes),
                "tissue": tissue,
                "age_mapping": AGE_MAPPING
            },
            model_path
        )

        # ----------------------------
        # plot internal clock
        # ----------------------------
        output_prefix = os.path.join(
            PLOT_DIR,
            f"{safe_tissue}__mlp_tanh_internal_scatter"
        )

        r, p, mae, axis_max = plot_internal_clock_scatter(
            tissue=tissue,
            y_test=y_test,
            y_pred=y_pred,
            output_prefix=output_prefix
        )

        summary_rows.append({
            "tissue": tissue,
            "model_name": "mlp_tanh",
            "model_path": model_path,
            "source_gene_file": gene_file,
            "base_sage_gene_count": len(selected_genes),
            "valid_sage_gene_count": len(valid_genes),
            "n_train": len(y_train),
            "n_test": len(y_test),
            "test_PCC": r,
            "test_P_value": p,
            "test_MAE": mae,
            "y_pred_min": float(np.min(y_pred)),
            "y_pred_max": float(np.max(y_pred)),
            "plot_axis_max": axis_max,
            "internal_scatter_pdf": output_prefix + ".pdf",
            "internal_scatter_png": output_prefix + ".png"
        })

    summary_df = pd.DataFrame(summary_rows)
    missing_df = pd.DataFrame(missing_rows)

    summary_path = os.path.join(
        SUMMARY_DIR,
        "Mouse_mlp_tanh_Internal_Clock_Summary.csv"
    )

    missing_path = os.path.join(
        SUMMARY_DIR,
        "Mouse_mlp_tanh_Missing_Mouse_Matrix_Genes.csv"
    )

    summary_df.to_csv(summary_path, index=False)
    missing_df.to_csv(missing_path, index=False)

    print(f"\nInternal clock summary saved: {summary_path}")
    print(f"Missing gene table saved: {missing_path}")

    return summary_df


# ============================================================
# 7. run
# ============================================================

if __name__ == "__main__":
    train_df = train_mouse_mlp_tanh_clocks()
    display(train_df)